# Grocery Basket Optimizer — v0

This notebook tests the first version of the grocery basket optimization logic.

The goal is to answer:

1. What is the cheapest single store for a user's grocery basket?
2. What is the cheapest two-store combination?
3. How much money does the user save by visiting two stores instead of one?

In [34]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current working directory:
/home/jovyan

Files/folders here:
['visualization_rules.ipynb', 'top_canadian_companies_2023.csv', 'top_20_companies.csv', 'text_analysis.ipynb', 'text_analysis-Copy1.ipynb', 'sql_overview.ipynb', 'shared', 'reviews.csv', 'regression_tree_Updated-2.ipynb', 'optimizer_v0.ipynb', 'ne_110m_admin_0_countries.shx', 'ne_110m_admin_0_countries.shp', 'ne_110m_admin_0_countries.prj', 'ne_110m_admin_0_countries.dbf', 'ne_110m_admin_0_countries.cpg', 'ne_110m_admin_0_countries.VERSION.txt', 'ne_110m_admin_0_countries.README.html', 'merge.ipynb', 'maps_renamed.ipynb', 'jbjdbrenj.ipynb', 'intro-to-web-scraping.ipynb', 'infoboxes.json', 'imdb_rendered.html', 'groupby.ipynb', 'grocery-basket-optimizer.zip', 'grocery-basket-optimizer', 'fortune_500_companies.csv', 'data_clean.ipynb', '__MACOSX', 'Untitled4.ipynb', 'Untitled3.ipynb', 'Untitled2.ipynb', 'Untitled1.ipynb', 'Untitled.ipynb', 'Untitled Folder', 'NYCAirbnb stuff.ipynb', 'HTML-based-scraping-2024.ipynb', 'API_based

In [35]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nParent folder:")
print(Path("..").resolve())

print("\nFiles in parent folder:")
for item in Path("..").iterdir():
    print(item)

Current working directory:
/home/jovyan

Parent folder:
/home

Files in parent folder:
../rstudio
../jovyan


In [36]:
import pandas as pd

prices = pd.read_csv("grocery-basket-optimizer/data/sample_prices.csv")
basket = pd.read_csv("grocery-basket-optimizer/data/sample_basket.csv")

prices.head()


,item,category,store,price,unit,package_size,flyer_week
0,eggs,dairy,No Frills,3.99,dozen,12,2026-07-08
1,eggs,dairy,Walmart,4.29,dozen,12,2026-07-08
2,eggs,dairy,Food Basics,3.79,dozen,12,2026-07-08
3,milk,dairy,No Frills,5.89,4L,4,2026-07-08
4,milk,dairy,Walmart,5.69,4L,4,2026-07-08


In [37]:
print(prices.shape)
print(basket.shape)
basket.head()


(45, 7)
(10, 2)


,item,quantity
0,eggs,1
1,milk,1
2,bread,2
3,chicken breast,1
4,pasta,2


In [38]:
basket_prices = basket.merge(prices, on="item", how="left")

basket_prices["total_item_cost"] = basket_prices["quantity"] * basket_prices["price"]

basket_prices.head(10)

,item,quantity,category,store,price,unit,package_size,flyer_week,total_item_cost
0,eggs,1,dairy,No Frills,3.99,dozen,12,2026-07-08,3.99
1,eggs,1,dairy,Walmart,4.29,dozen,12,2026-07-08,4.29
2,eggs,1,dairy,Food Basics,3.79,dozen,12,2026-07-08,3.79
3,milk,1,dairy,No Frills,5.89,4L,4,2026-07-08,5.89
4,milk,1,dairy,Walmart,5.69,4L,4,2026-07-08,5.69
5,milk,1,dairy,Food Basics,5.99,4L,4,2026-07-08,5.99
6,bread,2,bakery,No Frills,2.99,loaf,1,2026-07-08,5.98
7,bread,2,bakery,Walmart,2.79,loaf,1,2026-07-08,5.58
8,bread,2,bakery,Food Basics,3.19,loaf,1,2026-07-08,6.38
9,chicken breast,1,meat,No Frills,12.99,kg,1,2026-07-08,12.99


In [39]:
missing_prices = basket_prices[basket_prices["price"].isna()]

missing_prices

,item,quantity,category,store,price,unit,package_size,flyer_week,total_item_cost


In [40]:
single_store_costs = (
    basket_prices
    .groupby("store", as_index=False)["total_item_cost"]
    .sum()
    .sort_values("total_item_cost")
)

single_store_costs

,store,total_item_cost
0,Food Basics,53.37
1,No Frills,54.97
2,Walmart,57.17


In [41]:
best_single_store = single_store_costs.iloc[0]

print("Best single-store option:")
print(f"{best_single_store['store']} — ${best_single_store['total_item_cost']:.2f}")

Best single-store option:
Food Basics — $53.37


In [42]:
from itertools import combinations

def calculate_two_store_combo_cost(store_1, store_2, basket_prices):
    """
    For a given pair of stores, assign each basket item to the cheaper store.
    Returns the total cost and item-level recommendations.
    """
    
    pair_data = basket_prices[basket_prices["store"].isin([store_1, store_2])].copy()
    
    item_recommendations = (
        pair_data
        .sort_values(["item", "total_item_cost"])
        .groupby("item", as_index=False)
        .first()
    )
    
    total_cost = item_recommendations["total_item_cost"].sum()
    
    return total_cost, item_recommendations

In [43]:
stores = prices["store"].unique()

combo_results = []

for store_1, store_2 in combinations(stores, 2):
    total_cost, recommendations = calculate_two_store_combo_cost(
        store_1, 
        store_2, 
        basket_prices
    )
    
    combo_results.append({
        "store_combo": f"{store_1} + {store_2}",
        "total_cost": total_cost
    })

two_store_costs = (
    pd.DataFrame(combo_results)
    .sort_values("total_cost")
)

two_store_costs

,store_combo,total_cost
2,Walmart + Food Basics,51.07
1,No Frills + Food Basics,52.17
0,No Frills + Walmart,53.87


In [44]:
best_two_store_combo = two_store_costs.iloc[0]

print("Best two-store option:")
print(f"{best_two_store_combo['store_combo']} — ${best_two_store_combo['total_cost']:.2f}")

Best two-store option:
Walmart + Food Basics — $51.07


In [45]:
single_store_total = best_single_store["total_item_cost"]
two_store_total = best_two_store_combo["total_cost"]

savings = single_store_total - two_store_total

print(f"Best single-store cost: ${single_store_total:.2f}")
print(f"Best two-store cost: ${two_store_total:.2f}")
print(f"Savings from visiting two stores: ${savings:.2f}")

Best single-store cost: $53.37
Best two-store cost: $51.07
Savings from visiting two stores: $2.30


In [46]:
best_store_1, best_store_2 = best_two_store_combo["store_combo"].split(" + ")

best_two_store_total, best_item_recommendations = calculate_two_store_combo_cost(
    best_store_1,
    best_store_2,
    basket_prices
)

best_item_recommendations[
    ["item", "quantity", "store", "price", "total_item_cost"]
].sort_values(["store", "item"])

,item,quantity,store,price,total_item_cost
0,apples,1,Food Basics,4.49,4.49
3,cheese,1,Food Basics,4.99,4.99
4,chicken breast,1,Food Basics,11.99,11.99
5,eggs,1,Food Basics,3.79,3.79
8,pasta,2,Food Basics,1.79,3.58
9,yogurt,1,Food Basics,3.79,3.79
1,bananas,1,Walmart,1.59,1.59
2,bread,2,Walmart,2.79,5.58
6,frozen vegetables,2,Walmart,2.79,5.58
7,milk,1,Walmart,5.69,5.69


In [47]:
if savings < 3:
    recommendation = "The two-store option saves very little, so the best practical choice is probably the cheapest single store."
elif savings < 10:
    recommendation = "The two-store option saves a moderate amount. It may be worth it if the stores are close together."
else:
    recommendation = "The two-store option creates meaningful savings and may be worth the extra trip."

print("Recommendation:")
print(recommendation)

Recommendation:
The two-store option saves very little, so the best practical choice is probably the cheapest single store.


In [48]:
print("GROCERY BASKET OPTIMIZER SUMMARY")
print("--------------------------------")
print(f"Best single-store option: {best_single_store['store']} — ${single_store_total:.2f}")
print(f"Best two-store option: {best_two_store_combo['store_combo']} — ${two_store_total:.2f}")
print(f"Savings from two stores: ${savings:.2f}")
print()
print(recommendation)

GROCERY BASKET OPTIMIZER SUMMARY
--------------------------------
Best single-store option: Food Basics — $53.37
Best two-store option: Walmart + Food Basics — $51.07
Savings from two stores: $2.30

The two-store option saves very little, so the best practical choice is probably the cheapest single store.
